In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from worker.app.pdf_text import PDFTextExtractor

raw_pdf_dir = PROJECT_ROOT / "data" / "samples" / "raw_pdfs"
extracted_text_dir = PROJECT_ROOT / "data" / "samples" / "extracted_text"

pdf_path = sorted(raw_pdf_dir.glob("*.pdf"))[0]

extractor = PDFTextExtractor()
result = extractor.extract(pdf_path)

print("file:", result.pdf_path.name)
print("page_count:", result.page_count)
print("pages_with_text:", result.pages_with_text)
print("total_chars:", result.total_chars)
print("avg_chars_per_page:", result.avg_chars_per_page)
print("quality:", result.extraction_quality)
print("flags:", result.quality_flags)

print(result.text[:2000])

file: 1010_1120_HUD_AFGE-_redacted.pdf
page_count: 284
pages_with_text: 284
total_chars: 684988
avg_chars_per_page: 2411.9295774647885
quality: good
flags: []
Agreement
between
U.S. Department of Housing
and Urban Development
and
American Federation of
Government Employees
AFL-CIO

TABLE OF CONTENTS
NUMBER ARTICLE TITLE PAGE
TABLE OF CONTENTS ........................................................................................ 1
PREAMBLE ....................................................................................................... - 5 -
INTRODUCTION
ARTICLE 1 COVERAGE AND RECOGNITION ................................................................. - 6 -
ARTICLE 2 DEFINITIONS .................................................................................................... - 8 -
LABOR-MANAGEMENT COLLABORATION
ARTICLE 3 LABOR MANAGEMENT FORM ................................................................... - 11 -
ARTICLE 4 RIGHTS AND OBLIGATIONS OF THE PARTIES...........

In [3]:
import pandas as pd

rows = []

for pdf_path in sorted(raw_pdf_dir.glob("*.pdf")):
    result = extractor.extract(pdf_path)
    extractor.save_text(result, extracted_text_dir)

    rows.append(
        {
            "filename": pdf_path.name,
            "page_count": result.page_count,
            "pages_with_text": result.pages_with_text,
            "text_page_ratio": round(result.pages_with_text / result.page_count, 3),
            "total_chars": result.total_chars,
            "avg_chars_per_page": round(result.avg_chars_per_page, 2),
            "quality": result.extraction_quality,
            "flags": ", ".join(result.quality_flags),
        }
    )

quality_df = pd.DataFrame(rows)
quality_df

,filename,page_count,pages_with_text,text_page_ratio,total_chars,avg_chars_per_page,quality,flags
0,1010_1120_HUD_AFGE-_redacted.pdf,284,284,1.000,684988,2411.93,good,
1,1012_DOI_BIA_Indian_Educators_Federation_4524_...,294,294,1.000,549612,1869.43,good,
2,1085_FMCS_NAGE_R3-118_11102020-redacted.pdf,71,71,1.000,115053,1620.46,good,
3,1501_Treasury_BEP_IPPDE_Local_32_Engravers_051...,62,62,1.000,100746,1624.94,good,
4,1555_1564_DOE_AFGE_788_06302019-_redacted.pdf,156,156,1.000,300909,1928.90,good,
5,2376_Treasury_Mint_FOP_CBA_updated_Redacted.pdf,116,113,0.974,175527,1513.16,good,
6,2672_DOI_NPS_Independence_NHP_AFGE_2058_081220...,166,154,0.928,327208,1971.13,good,
7,2679_DOT_SLSDC_AFGE_1968_09302021-redacted.pdf,108,108,1.000,172071,1593.25,good,
8,3708_Army_USACE_CBA_02022020.pdf,69,69,1.000,145199,2104.33,good,
9,ARS_Animal_Disease_Center2015-06-29BUS_1529.pdf,115,115,1.000,221336,1924.66,good,


In [4]:
quality_report_path = PROJECT_ROOT / "data" / "sample_sources" / "pdf_text_extraction_quality.csv"
quality_df.to_csv(quality_report_path, index=False)
quality_report_path

PosixPath('/Users/mikifoster/Documents/Code_Projects/cba-clock/data/sample_sources/pdf_text_extraction_quality.csv')

In [5]:
from worker.app.section_extraction import ContractSectionExtractor

text_dir = PROJECT_ROOT / "data" / "samples" / "extracted_text"
text_path = sorted(text_dir.glob("*.txt"))[0]

section_extractor = ContractSectionExtractor()
sections = section_extractor.extract_sections(text_path)

len(sections)

58

In [6]:
sections_df = pd.DataFrame(
    [
        {
            "source_file": section.source_file,
            "article_number": section.article_number,
            "article_title": section.article_title,
            "start_char": section.start_char,
            "end_char": section.end_char,
            "char_count": len(section.text),
        }
        for section in sections
    ]
)

sections_df.head(30)

,source_file,article_number,article_title,start_char,end_char,char_count
0,1010_1120_HUD_AFGE-_redacted.txt,1,COVERAGE AND RECOGNITION,9500,13745,4243
1,1010_1120_HUD_AFGE-_redacted.txt,2,DEFINITIONS,13745,20372,6625
2,1010_1120_HUD_AFGE-_redacted.txt,3,LABOR MANAGEMENT FORUM/RELATIONS MEETINGS,20372,26841,6467
3,1010_1120_HUD_AFGE-_redacted.txt,4,RIGHTS AND OBLIGATIONS OF THE PARTIES,26841,32926,6083
4,1010_1120_HUD_AFGE-_redacted.txt,5,DUES WITHHOLDING,32926,41278,8350
5,1010_1120_HUD_AFGE-_redacted.txt,6,EMPLOYEE RIGHTS/STANDARDS OF CONDUCT,41278,51567,10287
6,1010_1120_HUD_AFGE-_redacted.txt,7,PROFESSIONAL EMPLOYEES,51567,52840,1271
7,1010_1120_HUD_AFGE-_redacted.txt,8,TEMPORARY EMPLOYEES,52840,54525,1683
8,1010_1120_HUD_AFGE-_redacted.txt,9,Equal Employment Opportunity and Discriminatio...,54525,72717,18190
9,1010_1120_HUD_AFGE-_redacted.txt,10,COLLABORATIVE CONFLICT RESOLUTION (CCR),72717,78369,5650


## Extraction Plan
Layer 1: section map
- article number
- article title
- start/end location
- full section text

Layer 2: candidate rule extraction
- grievance/procedure/time-limit rules first
- then discipline/performance/probation rules

Layer 3: structured rule typing
- deadline rule
- applicability rule
- exclusion rule
- escalation rule
- remedy rule
- representation rule

In [7]:
all_section_rows = []

for text_path in sorted(extracted_text_dir.glob("*.txt")):
    sections = section_extractor.extract_sections(text_path)

    for section in sections:
        all_section_rows.append(
            {
                "article_title": section.article_title.upper(),
            }
        )

all_sections_df = pd.DataFrame(all_section_rows)

unique_titles = (
    all_sections_df["article_title"]
    .drop_duplicates()
    .sort_values()
)

unique_titles.tolist()

['ABSENCE AND LEAVE',
 'ABSENT WITHOUT OFFICIAL LEAVE/ENFORCED LEAVE',
 'ACTIONS BASED ON UNACCEPTABLE PERFORMANCE',
 'ADDITIONAL DUES ALLOTMENT',
 'ADMINISTRATIVE ELECTRONIC MAIL & SOCIAL MEDIA',
 'ADMINISTRATIVE LEAVE',
 'ADVERSE ACTIONS',
 'AFFIRMATIVE ACTION',
 'ALTERNATIVE DISPUTE RESOLUTION (ADR)',
 'ANNUAL LEAVE',
 'ARBITRATION',
 'ARBITRATION PROCEDURE',
 'ASSIGNMENTS AND DETAILS',
 'ATTENDANCE AND LEAVE',
 'AWARDS AND RECOGNITION',
 'AWARDS PROGRAM',
 'BASIC WORKWEEK',
 'BASIC WORKWEEK AND OVERTIME',
 'BREAK ROOMS/BREAK AREAS',
 'CALENDAR DAYS',
 'CAREER SEASONAL POSITIONS',
 'CHANGES IN CONDITIONS OF EMPLOYMENT/UNION',
 'CHILD CARE/ELDER CARE',
 'COLLABORATIVE CONFLICT RESOLUTION (CCR)',
 'COMMITTEES',
 'CONTRACTING OUT',
 'CONTRACTING OUT BARGAINING UNIT WORK',
 'CONTRACTING OUT WORK',
 'COURT LEAVE',
 'COVERAGE AND RECOGNITION',
 'CRITICAL INCIDENT STRESS DEBRIEFING',
 'DAY CARE',
 'DEFINITIONS',
 'DETAILS',
 'DETAILS AND REASSIGNMENTS',
 'DETAILS AND TEMPORARY PROMOTIONS',

In [8]:
article_inventory_path = PROJECT_ROOT / "data" / "sample_sources" / "article_title_inventory.csv"

article_title_inventory = (
    all_sections_df["article_title"]
    .drop_duplicates()
    .sort_values()
    .to_frame(name="article_title")
)

article_title_inventory.to_csv(article_inventory_path, index=False)
article_inventory_path

PosixPath('/Users/mikifoster/Documents/Code_Projects/cba-clock/data/sample_sources/article_title_inventory.csv')

In [9]:
taxonomy_template_path = PROJECT_ROOT / "data" / "sample_sources" / "article_title_taxonomy_template.csv"

taxonomy_template = article_title_inventory.copy()
taxonomy_template["domain_area"] = ""
taxonomy_template["rule_relevance"] = ""
taxonomy_template["notes"] = ""

taxonomy_template.to_csv(taxonomy_template_path, index=False)
taxonomy_template_path

PosixPath('/Users/mikifoster/Documents/Code_Projects/cba-clock/data/sample_sources/article_title_taxonomy_template.csv')

In [13]:
import re
import pandas as pd

taxonomy_path = PROJECT_ROOT / "data" / "sample_sources" / "article_title_taxonomy_template.csv"

taxonomy_df = pd.read_csv(taxonomy_path)

DOMAIN_RULES = [
    ("dispute_resolution", r"GRIEVANCE|ARBITRATION|ALTERNATIVE DISPUTE|ADR|COLLABORATIVE CONFLICT|UNFAIR LABOR PRACTICE"),
    ("discipline_adverse_actions", r"DISCIPLIN|ADVERSE ACTION|LAST CHANCE|INVESTIGATION|WARNING|UNACCEPTABLE PERFORMANCE|PERFORMANCE-BASED ACTION"),
    ("performance_management", r"PERFORMANCE APPRAIS|PERFORMANCE EVALUATION|PERFORMANCE MANAGEMENT|PERFORMANCE STANDARDS|WITHIN-GRADE"),
    ("staffing_career_movement", r"MERIT PROMOTION|PROMOTION|INTERNAL PLACEMENT|DETAIL|TEMPORARY PROMOTION|REASSIGNMENT|LATERAL MOVEMENT|STAFFING|POSITION CLASSIFICATION|POSITION DESCRIPTION|UPWARD MOBILITY|CAREER SEASONAL|PROBATIONARY|TEMPORARY EMPLOYEES|REDUCTION|RIF|TRANSFER OF FUNCTION|SENIORITY|TESTS AND EMPLOYEE SELECTION"),
    ("training_development", r"TRAINING|CAREER DEVELOPMENT|EMPLOYEE DEVELOPMENT|EDUCATION|APPRENTICESHIP|EDUCATIONAL REIMBURSEMENT"),
    ("leave_attendance", r"LEAVE|ABSENCE|AWOL|ATTENDANCE|FAMILY AND MEDICAL|HOLIDAYS|TARDINESS|NON PAY STATUS"),
    ("work_schedule_overtime", r"HOURS OF WORK|HOURS OF DUTY|WORKWEEK|OVERTIME|COMPENSATORY|FLEXTIME|ALTERNATIVE WORK|BASIC WORKWEEK|SUNDAY PREMIUM|SHIFT DIFFERENTIAL|TELEWORK|FLEXIPLACE"),
    ("union_structure_representation", r"UNION RIGHTS|UNION REPRESENTATION|OFFICIAL TIME|DUES|ALLOTMENT|LABOR MANAGEMENT|LABOR-MANAGEMENT|RECOGNITION AND UNIT|COVERAGE AND RECOGNITION|BARGAINING UNIT|NAMES OF EMPLOYEES|ORIENTATION OF NEW EMPLOYEES|RIGHTS AND OBLIGATIONS"),
    ("bargaining_contract_term", r"DURATION|TERMINATION|MID-TERM|MIDCONTRACT|SUCCESSOR|NEGOTIATION|SUPPLEMENTATION|CHANGES IN CONDITIONS|LOCAL SUPPLEMENTS|NO STRIKE|STRIKES AND PICKETING"),
    ("pay_awards_benefits", r"PAY|WAGES|AWARDS|RECOGNITION|INCENTIVE|STUDENT LOAN|TRANSIT|FARE SUBSID|RETIREMENT|BENEFITS"),
    ("health_safety_accommodation", r"SAFETY|HEALTH|WELLNESS|WORKERS COMPENSATION|INJURY|REASONABLE ACCOMMODATION|FITNESS FOR DUTY|MEDICAL|HAZARDOUS|EAP|EMPLOYEE ASSISTANCE|SUBSTANCE TESTING|DRUG-FREE|LIGHT DUTY|SAFE WORKING CONDITIONS"),
    ("facilities_equipment_workplace", r"FACILITIES|EQUIPMENT|LOCKER|BREAK ROOM|PARKING|TRANSPORTATION|TRAVEL|GOVERNMENT FURNISHED|PROPERTY|OFFICE EQUIPMENT|EMAIL|SOCIAL MEDIA|SPACE MANAGEMENT|UNIFORM|CLOTHING|FOOTWEAR|EYEGLASSES|WORKPLACE|SMOKING|SMOKE FREE|REST BREAK"),
    ("records_privacy_admin", r"RECORDS|PRIVACY|EMPLOYEE INFORMATION|INFORMATION AND PUBLICATIONS|PRINTING|DISTRIBUTION|DEFINITIONS|NOTIFICATION PROCEDURES|CALENDAR DAYS"),
    ("management_rights_operations", r"MANAGEMENT RIGHTS|EMPLOYER RIGHTS|CONTRACTING OUT|SURVEILLANCE|USE OF FORCE|LAW ENFORCEMENT|MOTOR VEHICLE|FIREARMS|CRITICAL INCIDENT|OCCUPANT EMERGENCY|PANDEMIC|CONTINUITY"),
    ("legal_compliance", r"EQUAL EMPLOYMENT|EEO|AFFIRMATIVE ACTION|PROHIBITED PERSONNEL|LAW AND REGULATION|EFFECT OF LAW|PROVISIONS OF LAW|CHILD CARE|ELDER CARE|OUTSIDE EMPLOYMENT"),
]


def classify_domain(title: str) -> str:
    normalized = str(title).upper()

    for domain_area, pattern in DOMAIN_RULES:
        if re.search(pattern, normalized):
            return domain_area

    return "other"


taxonomy_df["domain_area"] = taxonomy_df["article_title"].apply(classify_domain)

if "rule_relevance" in taxonomy_df.columns:
    taxonomy_df = taxonomy_df.drop(columns=["rule_relevance"])

taxonomy_df.to_csv(taxonomy_path, index=False)
taxonomy_df

,article_title,domain_area,notes
0,ABSENCE AND LEAVE,leave_attendance,NaN
1,ABSENT WITHOUT OFFICIAL LEAVE/ENFORCED LEAVE,leave_attendance,NaN
2,ACTIONS BASED ON UNACCEPTABLE PERFORMANCE,discipline_adverse_actions,NaN
3,ADDITIONAL DUES ALLOTMENT,union_structure_representation,NaN
4,ADMINISTRATIVE ELECTRONIC MAIL & SOCIAL MEDIA,facilities_equipment_workplace,NaN
...,...,...,...
227,WITHIN-GRADE PAY INCREASES,performance_management,NaN
228,WORK CLOTHES/FOOTWEAR/EYEGLASSES,facilities_equipment_workplace,NaN
229,WORKERS COMPENSATION,health_safety_accommodation,NaN
230,WORKPLACE OF THE FUTURE,facilities_equipment_workplace,NaN


In [16]:
taxonomy_df[taxonomy_df["domain_area"] == "other"]

,article_title,domain_area,notes
24,COMMITTEES,other,NaN
31,DAY CARE,other,NaN
57,EMPLOYEE COUNSELING SERVICES PROGRAM,other,NaN
64,EMPLOYEE RIGHTS,other,NaN
65,EMPLOYEE RIGHTS/STANDARDS OF CONDUCT,other,NaN
66,EMPLOYEE VOLUNTEER PROVISIONS,other,NaN
68,EMPLOYER/UNION COOPERATION,other,NaN
80,FURLOUGHS FOR THIRTY (30) DAYS OR LESS,other,NaN
100,LABOR COMMITTEE CHAIRMAN/CHIEF OF POLICE,other,NaN
123,MULTILINGUAL EMPLOYEES,other,NaN
